# PQAX footprint of a lakehouse

Collects the footprint of the Delta tables of the default lakehouse of the notebook, or of items of a workspace, with `python -m pqax collect`, as the identity that runs the notebook. It keeps the file in the `Files` area of the default lakehouse, and exports its tables and their measures as Delta tables into the lakehouse, where a Direct Lake model can read them. It reads the tables and writes only the footprint and the export. See `templates/fabric/README.md` in https://github.com/pqax-org/pqax.

Attach a default lakehouse, set the parameters, and run all. A pipeline can pass the parameters to the parameter cell.

## Install the package

The Python package `pqax` with its extras `collect` (pyarrow, for the Delta checkpoints) and `delta` (deltalake, for the export). The version is 0.2.2, the first whose export writes over the mount of the lakehouse: change it to a later one when there is one. The cell comes first, since `%pip` can restart the Python of the notebook.

In [ ]:
%pip install --quiet "pqax[collect,delta]==0.2.2"

In [ ]:
SOURCE = "lakehouse"                             # "lakehouse": the default lakehouse of the notebook, read where it is mounted;
                                                 # "workspace": the items below, read from OneLake with the token of the notebook
WORKSPACE = ""                                   # with "workspace": the name of the workspace; empty for the workspace of the notebook
ITEMS = ""                                       # with "workspace": the items, separated by commas, such as "Sales.Lakehouse, Finance.Warehouse"
MOUNT = "/lakehouse/default"                     # with "lakehouse": where the default lakehouse is mounted
OUTPUT_FOLDER = "/lakehouse/default/Files/footprints"   # where the footprint goes
EXPORT_TO = "/lakehouse/default/Tables"          # where the export goes, as Delta tables; empty for no export
EXTRA_OPTIONS = ""                               # more options of pqax collect, separated by spaces, such as "--level 1"

## Collect

With `SOURCE = "lakehouse"`, `pqax collect --mount` reads the default lakehouse in the folder where Fabric mounts it, `/lakehouse/default`, which has no item level: `--item` gives it its name, from the context of the notebook. With `SOURCE = "workspace"`, it reads the items from OneLake over the DFS endpoint, with the token that `notebookutils.credentials.getToken("storage")` gives for `https://storage.azure.com`: the token of the user in an interactive session, of the owner of the schedule or of the pipeline otherwise. The command reads it from an environment variable of its own process, `--access-token-env`, so the token is not on a command line and is not printed. A token lasts about an hour: a collection that lasts longer fails with a 401.

In [ ]:
import datetime
import os
import re
import subprocess
import sys
import tempfile


def pqax(*arguments, env=None):
    """Runs python -m pqax with the arguments, prints what it prints, and returns the exit code."""
    completed = subprocess.run([sys.executable, "-m", "pqax", *arguments], env=env or dict(os.environ),
                               capture_output=True, text=True)
    print(completed.stdout + completed.stderr)
    return completed.returncode


context = notebookutils.runtime.context
environment = dict(os.environ)
if SOURCE == "lakehouse":
    lakehouse = context.get("defaultLakehouseName")
    if not lakehouse:
        raise ValueError("Attach a default lakehouse to the notebook.")
    source = ["--mount", MOUNT, "--item", f"{lakehouse}.Lakehouse"]
    label = lakehouse
elif SOURCE == "workspace":
    workspace = WORKSPACE.strip() or context.get("currentWorkspaceName")
    items = [item.strip() for item in ITEMS.split(",") if item.strip()]
    if not workspace or not items:
        raise ValueError("Set WORKSPACE, or run the notebook in the workspace to collect, and ITEMS.")
    source = ["--workspace", workspace, "--access-token-env", "PQAX_ACCESS_TOKEN"]
    for item in items:
        source += ["--item", item]
    environment["PQAX_ACCESS_TOKEN"] = notebookutils.credentials.getToken("storage")
    label = workspace
else:
    raise ValueError('SOURCE is "lakehouse" or "workspace".')

slug = re.sub(r"[^a-z0-9]+", "-", label.lower()).strip("-") or "footprint"
name = f"{slug}-{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d}.pqax"
local = os.path.join(tempfile.gettempdir(), name)
exit_code = pqax("collect", *source, "--output", local, *EXTRA_OPTIONS.split(), env=environment)
environment.pop("PQAX_ACCESS_TOKEN", None)
if exit_code != 0:
    raise RuntimeError(f"pqax collect ended with exit code {exit_code}: 1 for a file that is not valid, 2 for a failure.")

## Check and keep the footprint

In [ ]:
import shutil

if pqax("check", local) != 0:
    raise RuntimeError(f"{name} is not valid.")
pqax("info", local)
pqax("tables", local, "--top", "20")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
footprint = os.path.join(OUTPUT_FOLDER, name)
shutil.copy(local, footprint)
print(f"{name} is in {OUTPUT_FOLDER}.")

## Export into the lakehouse

`pqax export --format delta` appends the tables of the footprint and their measures to Delta tables in `EXPORT_TO`, one table per name: `footprints`, `tableMeasures`, `columnMeasures`, `collectionMeasures`, and those of the format, such as `tables` and `files`. In `/lakehouse/default/Tables` they are tables of the lakehouse, which the Power BI template reads in Direct Lake. A footprint already in the export is skipped, so running the cell again adds nothing; each run of the notebook collects a new footprint, which the export adds. When the notebook collects its own default lakehouse, the next footprint includes these tables too.

In [ ]:
if EXPORT_TO.strip():
    if pqax("export", footprint, "--to", EXPORT_TO.strip(), "--format", "delta", "--if-exists", "skip") != 0:
        raise RuntimeError(f"The export of {name} failed.")

## Run it every day

A notebook has its own schedule: in the notebook, Run, Schedule, with a daily repetition. To pass parameters, or to collect several lakehouses or workspaces, create a pipeline with one Notebook activity per run of this notebook, set the parameters under Base parameters with the names of the parameter cell, and give the pipeline a schedule trigger, such as every day at 04:00. The activities that write into the same `EXPORT_TO` run one after the other: link them with On success, since two exports that append to the same Delta table at the same time can conflict. A scheduled run uses the identity that owns the schedule or the pipeline, so that identity needs a role that reads every item collected. The files of the lakehouse are not pruned: a later cell or a pipeline step can delete the footprints older than a number of days.